# 5단계: BERTopic 토픽모델링 (생활, 화장품)

**목적**: 4단계까지는 AIHub가 미리 정의한 속성(`Aspect`)으로만 분석했다. 이번 단계는 정해진 속성 틀 없이, **부정 리뷰**(`GeneralPolarity == -1`)들을 비지도로 클러스터링해서 새로운 불만 유형(토픽)을 발견한다.

**ML/DL 구분 (발표자료용)**

| 구성요소 | 역할 | 구분 |
|---|---|---|
| Sentence-BERT 임베딩 | 문장 벡터화 | DL — 2단계에서 이미 계산해둔 걸 재사용 (추론만, 학습 없음) |
| UMAP | 차원축소 | 비-DL (매니폴드 학습) |
| HDBSCAN | 클러스터링 | 비-DL (밀도기반 비지도 ML) |
| c-TF-IDF | 토픽 키워드 추출 | 비-DL (통계/정보검색) |

**입력**: 리뷰 전체가 아니라 부정 극성(`GeneralPolarity == -1`) 리뷰만 사용한다. 전체로 돌리면 "만족" 계열 토픽이 상위를 차지해서 결함 탐지에 안 쓰인다.

**도메인별 개별 모델**: 3·4단계와 동일하게 도메인마다 어휘가 달라서 따로 학습한다.

In [1]:
import pandas as pd
import numpy as np
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
from sentence_transformers import SentenceTransformer

reviews = pd.read_parquet("../data/processed/reviews.parquet")
embeddings = np.load("../data/processed/embeddings.npy")
embedding_ids = pd.read_parquet("../data/processed/embedding_ids.parquet")

# embeddings.npy의 행 순서가 reviews.parquet과 정확히 같다는 걸 전제로 위치(iloc) 인덱싱을 쓴다.
# 이 전제가 깨지면 완전히 다른 리뷰에 다른 임베딩이 매칭되는 조용한 버그가 생기므로 반드시 assert로 확인한다.
assert len(reviews) == len(embedding_ids) == embeddings.shape[0]
assert (reviews["review_id"].values == embedding_ids["review_id"].values).all(), \
    "reviews.parquet과 embedding_ids.parquet의 행 순서가 다름 -- 위치 인덱싱 불가"
print(f"reviews {len(reviews)}행, embeddings {embeddings.shape} -- 순서 일치 확인 완료")

C:\Users\user\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


reviews 225283행, embeddings (225283, 768) -- 순서 일치 확인 완료


In [2]:
# 2단계와 동일한 SBERT 모델을 명시적으로 지정 (기본값인 영어 모델이 쓰이는 걸 방지)
sbert = SentenceTransformer("jhgan/ko-sroberta-multitask")

# tokens(1단계 Kiwi 형태소분석 결과, 조사/어미 제거된 내용어)를 c-TF-IDF 키워드 추출 입력으로 쓴다.
# RawText_clean을 그대로 쓰면 "가격이"/"가격은"처럼 조사가 붙은 채로 쪼개져 키워드가 지저분해진다.
vectorizer_model = CountVectorizer(token_pattern=r"(?u)\b\w+\b")  # 1글자 한국어 단어도 살리기 위해 기본 2글자 제한 해제

DOMAINS = ["생활", "화장품"]
MIN_TOPIC_SIZE = 20

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:  96%|█████████▌| 191/199 [00:00<00:00, 1900.79it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1856.32it/s]

In [3]:
from gensim.corpora import Dictionary
from gensim.models.coherencemodel import CoherenceModel

def compute_coherence(topic_model, docs, top_n=10):
    """NPMI coherence: 토픽 상위 단어들이 실제로 같이 등장하는 경향이 있는지를 정량화한 지표.
    -1~1 범위, 높을수록 토픽 내 단어들이 서로 의미적으로 밀접하다는 뜻.
    지금까지는 사람이 키워드+대표리뷰를 읽고 눈으로만 검증했는데, 이번에 정량 지표를 추가한다."""
    texts = [d.split() for d in docs]
    dictionary = Dictionary(texts)
    topic_ids = sorted([t for t in topic_model.get_topics().keys() if t != -1])
    topic_words = [[w for w, _ in topic_model.get_topic(tid)[:top_n]] for tid in topic_ids]
    cm = CoherenceModel(topics=topic_words, texts=texts, dictionary=dictionary, coherence="c_npmi")
    per_topic = dict(zip(topic_ids, cm.get_coherence_per_topic()))
    return cm.get_coherence(), per_topic

In [4]:
def run_bertopic_for_domain(domain: str):
    mask = (reviews["Domain"] == domain) & (reviews["GeneralPolarity"] == -1)
    sub = reviews.loc[mask]
    idx = sub.index.to_numpy()  # reviews.parquet이 index=False로 저장돼 있어 기본 RangeIndex = embeddings.npy 행 위치와 같음
    docs = sub["tokens"].tolist()
    emb_sub = embeddings[idx]

    print(f"[{domain}] 부정 리뷰 {len(sub)}건으로 BERTopic 학습 시작")

    topic_model = BERTopic(
        embedding_model=sbert,
        vectorizer_model=vectorizer_model,
        min_topic_size=MIN_TOPIC_SIZE,
        calculate_probabilities=False,
        low_memory=True,  # 로컬 메모리가 빠듯해서(UMAP 최근접이웃 탐색 중 MemoryError 발생) 켬
        verbose=False,
    )
    topics, _ = topic_model.fit_transform(docs, embeddings=emb_sub)

    n_outlier_before = sum(1 for t in topics if t == -1)
    # outlier(-1, 어느 토픽에도 안 묶인 리뷰)를 c-TF-IDF 유사도 기준으로 가장 가까운 토픽에 재배정.
    # (1차 결과에서 outlier 비율이 32~39%로 높았던 걸 다듬는 작업 -- 새 클러스터링을 다시 하는 게
    # 아니라, 이미 만들어진 토픽 중 가장 비슷한 곳에 애매한 리뷰들을 마저 배정하는 후처리)
    new_topics = topic_model.reduce_outliers(docs, topics, strategy="c-tf-idf")
    topic_model.update_topics(docs, topics=new_topics)
    n_outlier_after = sum(1 for t in new_topics if t == -1)
    print(f"[{domain}] outlier 재배정: {n_outlier_before}건 -> {n_outlier_after}건")

    result = pd.DataFrame({
        "review_id": sub["review_id"].values,
        "topic_id": new_topics,
    })
    kw_map = {}
    for tid in result["topic_id"].unique():
        if tid == -1:
            continue
        words = [w for w, _ in topic_model.get_topic(tid)[:8]]
        kw_map[tid] = ", ".join(words)
    result["topic_keywords"] = result["topic_id"].map(kw_map).fillna("(outlier)")

    return topic_model, result, sub

## 생활

In [5]:
model_생활, result_생활, sub_생활 = run_bertopic_for_domain("생활")

n_total = len(result_생활)
n_outlier = (result_생활["topic_id"] == -1).sum()
n_topics = result_생활["topic_id"].nunique() - (1 if -1 in result_생활["topic_id"].values else 0)
print(f"토픽 수(outlier 제외): {n_topics}")
print(f"outlier 비율: {n_outlier}/{n_total} ({n_outlier/n_total*100:.1f}%)")
print()
print(model_생활.get_topic_info().head(10))

[생활] 부정 리뷰 5322건으로 BERTopic 학습 시작


2026-08-16 22:47:09,936 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


[생활] outlier 재배정: 1554건 -> 0건
토픽 수(outlier 제외): 32
outlier 비율: 0/5322 (0.0%)

   Topic  Count              Name  \
0      0    661     0_세제_청소_세정_지우   
1      1    594   1_마스크_얼굴_사이즈_불편   
2      2    461     2_칫솔_잇몸_닦이_양치   
3      3    512     3_진하_강하_머리_지속   
4      4    388  4_냄비_코팅_후라이팬_손잡이   
5      5    197     5_튼튼_조립_빨래_부분   
6      6    146    6_휴지_화장지_먼지_감기   
7      7    259     7_빠르_배송_감사_가격   
8      8    196     8_포장_박스_교환_상품   
9      9    111    9_면도기_면도_수염_피부   

                                 Representation  \
0     [세제, 청소, 세정, 지우, 세척, 변기, 효과, 거품, 곰팡이, 사용]   
1    [마스크, 얼굴, 사이즈, 불편, 착용, 대형, 아프, 94, 중형, 너무]   
2     [칫솔, 잇몸, 닦이, 양치, 부드럽, 진동, 치약, 개운, 전동, 자극]   
3     [진하, 강하, 머리, 지속, 너무, 구매, 별로, 취향, 제품, 라벤더]   
4  [냄비, 코팅, 후라이팬, 손잡이, 스텐, 무겁, 기름, 인덕션, 요리, 사용]   
5    [튼튼, 조립, 빨래, 부분, 바퀴, 플라스틱, 약하, 고정, 다리, 보이]   
6    [휴지, 화장지, 먼지, 감기, 마트, 두께, 30, 날리, 차이, 부드럽]   
7     [빠르, 배송, 감사, 가격, 비싸, 저렴하, 아쉽, 구매, 생각, 너무]   
8      [포장, 박스, 교환, 상품, 불량, 그냥, 반품, 조립, 배송, 비닐]   
9   

In [6]:
# 눈으로 검증: 상위 5개 토픽의 키워드 + 원문 대표 리뷰 3건씩
top_topics = model_생활.get_topic_info().query("Topic != -1").sort_values("Count", ascending=False).head(5)
for _, row in top_topics.iterrows():
    tid = row["Topic"]
    print(f"=== 토픽 {tid} (리뷰 {row['Count']}건) ===")
    print("키워드:", ", ".join([w for w, _ in model_생활.get_topic(tid)[:8]]))
    review_ids = result_생활.loc[result_생활["topic_id"] == tid, "review_id"].head(3)
    examples = sub_생활[sub_생활["review_id"].isin(review_ids)]["RawText_clean"]
    for ex in examples:
        print("  -", ex[:80])
    print()

=== 토픽 0 (리뷰 661건) ===
키워드: 세제, 청소, 세정, 지우, 세척, 변기, 효과, 거품
  - 누구보다 빠르게 제품 써보고 후기 알려드리러 왔습니다. 오늘은 변기 클리너입니다. 저는 OO에서 나오는 다른 제품을 만족스럽게 써서 이번에도 기
  - 이웃님들 안녕하세요~~OO 바르는 곰팡이싹 입소문이 좋아 사용해 봤습니다. 내돈내산 솔직 리얼 리뷰 지금 시작합니다~~ 곰팡이제거에 특별한 효과
  - 간만에 세제를 바꿨습니다. 후기에 세정력 좋다고 난리 부르스 댓글들이 많아서 고민없이 구입했습니다. 지금부터 리얼후기 들어갑니다. 어제 제품을 

=== 토픽 1 (리뷰 594건) ===
키워드: 마스크, 얼굴, 사이즈, 불편, 착용, 대형, 아프, 94
  - 이웃님들~ 만나서 반가워요~ 지금부터 ㅇㅇ 마스크에 이야기를 해볼테니 집중해서 잘 읽어주시면 감사하겠습니다. 오프라인보다는 온라인이 확실히 가격
  - 오늘은 블로그를 만들고 처음 올리는 글입니다. 많이 떨리지만 이웃님들 보고 힘을 내도록 하겠습니다! 마스크는 다 거기서 거기다라는 생각이라서 가
  - 귀차니즘으로 인해 3주간 미뤄뒀던 후기를 지금부터 작성하려고 합니다. 새부리형은 접었다 폈다하면서 보관하기가 좋은 것 같아요. 다만 코부분이 얼

=== 토픽 3 (리뷰 512건) ===
키워드: 진하, 강하, 머리, 지속, 너무, 구매, 별로, 취향
  - 오늘도 솔직한 실사용 후기 들려드리러 왔습니다. 시작할게요~ 오늘 제품은 OO 섬유 유연제 중 보타닉 가든 향인데요. 초고농축이라 쓰여있어서 조
  - 이번에 리뉴얼된 OOO 섬유유연제입니다 냄새제거와 미세먼지까지 차단해 준다고 하니 사 용을 안 할 수가 없을것 같아 구매했는데 조금 별로라 실망
  - 흠.. 맨 전용이라 뭐가 특별한게 있나 싶어서 구매해봤는데.. 솔직 후기 ..그냥 바로 지금 시작해볼게요. 향이..약간 인위적이고 너무 진하더라

=== 토픽 2 (리뷰 461건) ===
키워드: 칫솔, 잇몸, 닦이, 양치, 부드럽, 진동, 

In [7]:
overall_생활, per_topic_생활 = compute_coherence(model_생활, sub_생활["tokens"].tolist())
print(f"생활 전체 NPMI coherence: {overall_생활:.4f}")
print()
ranked = sorted(per_topic_생활.items(), key=lambda x: x[1])
print("coherence 가장 낮은(불명확한) 토픽 5개:")
for tid, score in ranked[:5]:
    kw = ", ".join([w for w, _ in model_생활.get_topic(tid)[:6]])
    print(f"  토픽{tid} [{kw}]: {score:.4f}")
print("coherence 가장 높은(명확한) 토픽 5개:")
for tid, score in ranked[-5:][::-1]:
    kw = ", ".join([w for w, _ in model_생활.get_topic(tid)[:6]])
    print(f"  토픽{tid} [{kw}]: {score:.4f}")

생활 전체 NPMI coherence: 0.0752

coherence 가장 낮은(불명확한) 토픽 5개:
  토픽29 [가위, 오늘, 후기, 주방, 부분, 과일]: -0.0725
  토픽28 [비누, 거품, 무르, 기한, 유통, 단단]: -0.0718
  토픽19 [충전, 진동, 시리즈, 소음, 그립, 별로]: -0.0569
  토픽21 [매실, 장아찌, 제거, 실리콘, 짱아찌, 과육]: -0.0540
  토픽10 [물티슈, 티슈, 두께, 수분, 보습, 촉촉]: -0.0470
coherence 가장 높은(명확한) 토픽 5개:
  토픽9 [면도기, 면도, 수염, 피부, 자극, 깎이]: 0.2273
  토픽20 [에어컨, 냄새, 뿌리, 청소, 곰팡이, 퀴퀴]: 0.2109
  토픽2 [칫솔, 잇몸, 닦이, 양치, 부드럽, 진동]: 0.1936
  토픽22 [만들, 번거롭, 리필, 일일이, 가성비, 제습제]: 0.1894
  토픽1 [마스크, 얼굴, 사이즈, 불편, 착용, 대형]: 0.1644


## 화장품

In [8]:
model_화장품, result_화장품, sub_화장품 = run_bertopic_for_domain("화장품")

n_total = len(result_화장품)
n_outlier = (result_화장품["topic_id"] == -1).sum()
n_topics = result_화장품["topic_id"].nunique() - (1 if -1 in result_화장품["topic_id"].values else 0)
print(f"토픽 수(outlier 제외): {n_topics}")
print(f"outlier 비율: {n_outlier}/{n_total} ({n_outlier/n_total*100:.1f}%)")
print()
print(model_화장품.get_topic_info().head(10))

[화장품] 부정 리뷰 8808건으로 BERTopic 학습 시작


2026-08-16 22:47:34,932 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


[화장품] outlier 재배정: 3423건 -> 0건
토픽 수(outlier 제외): 41
outlier 비율: 0/8808 (0.0%)

   Topic  Count            Name  \
0      0   1985  0_염색_머리_머릿결_두피   
1      1    646  1_색상_입술_립스틱_바르   
2      2    644   2_향수_지속_뿌리_강하   
3      3    402   3_주름_효과_개선_눈가   
4      4    194  4_눈썹_그리_속눈썹_번지   
5      5    170  5_엄마_세트_어머니_선물   
6      6    138   6_남편_신랑_인원_사용   
7      7    226   7_겨울_건조_보습_날씨   
8      8    170  8_거품_클렌징_세정_세안   
9      9    293   9_비싸_가격_구매_품질   

                                     Representation  \
0        [염색, 머리, 머릿결, 두피, 냄새, 새치, 빠지, 염색약, 샴푸, 심하]   
1         [색상, 입술, 립스틱, 바르, 예쁘, 립밤, 매트, 틴트, 아쉽, 진하]   
2         [향수, 지속, 뿌리, 강하, 약하, 너무, 잔향, 좋아하, 냄새, 아쉽]   
3          [주름, 효과, 개선, 눈가, 아직, 모르, 관리, 바르, 촉촉, 팔자]   
4  [눈썹, 그리, 속눈썹, 번지, 마스카라, 붙이, 지우, 떨어지, 아이라이너, 라이너]   
5         [엄마, 세트, 어머니, 선물, 주름, 주문, 나누, 더블, 구성, 같이]   
6         [남편, 신랑, 인원, 사용, 피부, 바르, 아빠, 아쉽, 구매, 싫어하]   
7          [겨울, 건조, 보습, 날씨, 촉촉, 피부, 사용, 요즘, 제품, 바르]   
8        [거품, 클렌징, 세정, 세안, 지우, 씻기, 오일, 느

In [9]:
top_topics = model_화장품.get_topic_info().query("Topic != -1").sort_values("Count", ascending=False).head(5)
for _, row in top_topics.iterrows():
    tid = row["Topic"]
    print(f"=== 토픽 {tid} (리뷰 {row['Count']}건) ===")
    print("키워드:", ", ".join([w for w, _ in model_화장품.get_topic(tid)[:8]]))
    review_ids = result_화장품.loc[result_화장품["topic_id"] == tid, "review_id"].head(3)
    examples = sub_화장품[sub_화장품["review_id"].isin(review_ids)]["RawText_clean"]
    for ex in examples:
        print("  -", ex[:80])
    print()

=== 토픽 0 (리뷰 1985건) ===
키워드: 염색, 머리, 머릿결, 두피, 냄새, 새치, 빠지, 염색약
  - 이 제품 저도 이번이 첫 구매이고, 오늘 처음 사용했어요. 한번 사용해 본 후기인데 별로라고 말씀 드리고 싶네용ㅜㅜ 옆머리 새치가 보여서 간단하
  - 왜 전문가에게 받으라고 하는 지 너무 잘 알겠습니다. 제가 똥손일까요? ㅎㅎ 혼자서 다운펌을 할 수 있다는 말에 바로 구매해서 사용해봤어요. 일
  - 요즘 셀프로 하는 제품이 늘어나고 있는 것 같아요. 저는 혼자 집에서 하는 걸 좋아해서 저도 구매해봤습니다. 사실 부드럽게 잘 발리는데 다운펌 

=== 토픽 1 (리뷰 646건) ===
키워드: 색상, 입술, 립스틱, 바르, 예쁘, 립밤, 매트, 틴트
  - 이번에 친구한테 선물로 받게되서 써보게 됐습니다. 처음에 봤을때는 디자인이 너무 귀여운 패키지와 용기가 깜찍해서 쓰기전에도 너무 기분이 좋더라구
  - 컬러가 다양하고 쓰임새 있어 보여서 구매해 본 립펜슬입니다. 총 11가지로 다양한 컬러가 출시되어 있어요. 앞으로 더 나올지도 모르죠. 몇 가지
  - 사고 싶은 색상이 다 품절이라 겨우 이 색상 샀네요. 저의 픽은 바로 버건디~ 사실 버건디 조금 고민했는데 ㅠㅠ 다른 색상 살 걸 그랬네요.. 

=== 토픽 2 (리뷰 644건) ===
키워드: 향수, 지속, 뿌리, 강하, 약하, 너무, 잔향, 좋아하
  - 인터넷에 파는 향수는 대부분 가품이라는 얘기를 본 적이 있는데 정말 그런 걸까요? OO OOO 오 드 뚜왈렛 사용후기예요. 인터넷으로 먼저 산 
  - 요새 모공 고민이 많은 저는 OOO라는 단어를 보자마자 이거다! 하고 구매해 봤어요~ 여러 가지 허브 추출물 성분들이 들어있어요~ 그래서 피지케
  - 향수가 필요해서 급하게 구매를 했는데 품질이 너무 좋지 않아서 후기를 작성 합니다,, 괜히 산 것 같아 정말 후회하는 제품이에요,, ㅠㅠ 이 향

=== 토픽 3 (리뷰 402건) ===
키워드: 주름, 효과, 개선, 눈가, 아직, 모르

In [10]:
overall_화장품, per_topic_화장품 = compute_coherence(model_화장품, sub_화장품["tokens"].tolist())
print(f"화장품 전체 NPMI coherence: {overall_화장품:.4f}")
print()
ranked = sorted(per_topic_화장품.items(), key=lambda x: x[1])
print("coherence 가장 낮은(불명확한) 토픽 5개:")
for tid, score in ranked[:5]:
    kw = ", ".join([w for w, _ in model_화장품.get_topic(tid)[:6]])
    print(f"  토픽{tid} [{kw}]: {score:.4f}")
print("coherence 가장 높은(명확한) 토픽 5개:")
for tid, score in ranked[-5:][::-1]:
    kw = ", ".join([w for w, _ in model_화장품.get_topic(tid)[:6]])
    print(f"  토픽{tid} [{kw}]: {score:.4f}")

화장품 전체 NPMI coherence: 0.0828

coherence 가장 낮은(불명확한) 토픽 5개:
  토픽35 [손톱, 발톱, 네일, 갈라지, 디자인, 접착력]: -0.1524
  토픽30 [퍼프, 모르, 묻어나, 효과, 볼륨, 컬링]: -0.1367
  토픽27 [면도, 쉐이빙폼, 거품, 쉐이빙, 피부, 자극]: -0.0521
  토픽6 [남편, 신랑, 인원, 사용, 피부, 바르]: -0.0142
  토픽34 [쇼핑, 홈쇼핑, 방송, 커버, oo, 광고]: 0.0203
coherence 가장 높은(명확한) 토픽 5개:
  토픽16 [용기, 뚜껑, 불량, 펌핑, 불편, 나오]: 0.2211
  토픽5 [엄마, 세트, 어머니, 선물, 주름, 주문]: 0.2178
  토픽3 [주름, 효과, 개선, 눈가, 아직, 모르]: 0.1981
  토픽17 [기한, 유통, 제조, 임박, 보내, 일자]: 0.1891
  토픽15 [배송, 샘플, 유통, 기한, 느리, 비싸]: 0.1827


## 저장

In [11]:
import os
os.makedirs("../data/processed/topic_datasets", exist_ok=True)

result_생활.to_parquet("../data/processed/topic_datasets/생활_topics.parquet", index=False)
result_화장품.to_parquet("../data/processed/topic_datasets/화장품_topics.parquet", index=False)
print("저장 완료: topic_datasets/생활_topics.parquet, topic_datasets/화장품_topics.parquet")

저장 완료: topic_datasets/생활_topics.parquet, topic_datasets/화장품_topics.parquet
